# Day 14 — Delta Lake Performance Optimization

## Part 1 — Understanding the small-file problem

Before and after file compaction
100 Small Files  ------------------------------------------------------. 3 OPTIMIZED file
- Spark may need to open and MANAGE all seperate file            -REWRITES SMALL FILES into one file while preserving tables' logical data 
- Increases file handling, task scheduling , cloud storage       - does NOT REMOE OLD FILES Uunless used with **VACCUM**


OPTIMIZE improves **physical file layout without changing the logical contents** of the Delta table.

### Step 1 — Generate test data

below code generates 100,000 sample transactions with 1,000 distinct customers.

In [0]:
from pyspark.sql import functions as F

base_df = (
    spark.range(0, 100000)
    .withColumn(
        "custId",
        (F.col("id") % 1000 + 1001).cast("int")
    )
    .withColumn(
        "amount",
        (F.rand(seed=42) * 1000).cast("double")
    )
    .withColumn(
        "serviceType",
        F.when(F.col("id") % 3 == 0, "Wireless")
         .when(F.col("id") % 3 == 1, "Fiber")
         .otherwise("Internet")
    )
    .withColumn(
        "transactionDate",
        F.date_add(
            F.lit("2026-01-01").cast("date"),
            (F.col("id") % 90).cast("int")
        )
    )
    .drop("id")
)

display(base_df.limit(10))

### Step 2 — Create multiple small files

**repartition(40)** --> to encourage multiple output files for this exercise.
Depending on Databricks automatic write optimization, the actual file count may differ.

In [0]:
table_name = ("workspace.customer360_bronze.transactions_optimize_day14")

(
    base_df
    .repartition(40)
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(table_name)
)

### Step 3 — Inspect Delta table details

In [0]:
%sql
DESCRIBE DETAIL
workspace.customer360_bronze.transactions_optimize_day14;



# Part 3 — Run OPTIMIZE
### Step 4 — Compact the Delta table

In [0]:
%sql
OPTIMIZE workspace.customer360_bronze.transactions_optimize_day14;

### Step 5 — Inspect the table again

check the table after OPTIMIZE- 40 files will be clubed into 1 single file. and also there would be reduction in the file size as well. 

In [0]:
%sql
DESCRIBE DETAIL workspace.customer360_bronze.transactions_optimize_day14;

In [0]:
%sql
DESCRIBE HISTORY workspace.customer360_bronze.transactions_optimize_day14;

# Part 4: Z-Ordering
Z-Ordering colocates related column values in the same files, improving the effectiveness of Delta Lake data skipping.

**before Z-ordering file**
```
file 1 --> 1001-1999
file2 --> 1001-1999
file 3 --> 1001-1999
file 4 --> 1001-1999
file 5 --> 1001-1999
```

**After Z ordering files**
````
file 1 --> 1001-1250
file 2 --> 1251-1500
file 3 --> 1501-1750
file 4 --> 1751-2000
````



### Step 6 — Run Z-Ordering

**NOTE**-
- Your table currently has only one active data file.
- Therefore, Z-Ordering may provide little or no query-performance benefit in this particular test because Spark cannot skip some files while reading others when there is only one file.
- The Z-Order operation may also **rewrite the file without increasing the number of active files.**
- For a meaningful performance comparison, we'd normally use a larger table with multiple files.

In [0]:
%sql
OPTIMIZE workspace.customer360_bronze.transactions_optimize_day14
ZORDER BY (custId);

In [0]:
%sql
DESCRIBE DETAIL workspace.customer360_bronze.transactions_optimize_day14;

In [0]:
%sql
DESCRIBE HISTORY
workspace.customer360_bronze.transactions_optimize_day14;

### Step 7 — Understand data skipping

- since data is z-ordered by custid, they are batched by custid. when you do custId=1500 it will go to that particular segment that that hold this range data and fetch thus SKipping the rread through all other files- DATA SKIPPING

**_Note_**- 
- JSON file store **COLUMN-WISE MIN/MAX and NOT ROW-WISE**. EACH COLUMN TRACK ITS OWN MINIMUM AND MAXIMUM across ALL rows.



In [0]:
%sql
EXPLAIN FORMATTED
SELECT *
FROM workspace.customer360_bronze.transactions_optimize_day14
WHERE custId = 1500;

# Part 5: Partitioning vs Z-Ordering vs Liquid Clustering
### Step 8 — Create a partitioned Delta table

**Partitioning**
Physically organizes data into directories based on column values, such as transactionDate.
Queries can avoid reading irrelevant partitions through partition pruning.


**Z-Ordering**
Reorganizes records within Delta data files to improve locality for selected columns, helping file-level data skipping.
Useful for selective queries on columns such as custId.


**Liquid Clustering**
A newer Delta table-layout approach that organizes data using clustering keys and supports evolving those keys as query patterns change.
It avoids some limitations of traditional static partitioning and Z-Ordering.

In [0]:
partitioned_table = ("workspace.customer360_bronze.transactions_partitioned_day14")

(
    base_df.write
    .format("delta")
    .mode("overwrite")
    .partitionBy("transactionDate")
    .saveAsTable(partitioned_table)
)

### Step 9 — Inspect partition information

In [0]:
%sql
DESCRIBE DETAIL
workspace.customer360_bronze.transactions_partitioned_day14;

### Step 10 — Test partition pruning

In [0]:
%sql
EXPLAIN FORMATTED
SELECT *
FROM workspace.customer360_bronze.transactions_partitioned_day14
WHERE transactionDate = '2026-01-15';

# Part 6: Delta Lake VACUUM, Retention & Time Travel

. **What does VACUUM do?**

- VACUUM permanently deletes data files that are no longer referenced by the current Delta table snapshot and meet the retention requirements.

By default, Delta Lake uses a **7-day retention threshold for VACUUM.**

This means obsolete files generally become eligible for removal only after the retention period.
OPTIMIZE vs VACUUM
OPTIMIZE	                    VACUUM
Compacts and reorganizes files	Deletes eligible obsolete files
Improves data layout	        Reclaims physical storage
Preserves older files initially	May make older versions unreadable
Creates a new Delta version when changes are committed	Does not rewrite active table records

Interview point: OPTIMIZE improves performance; VACUUM handles storage c

In [0]:
%sql
VACUUM workspace.customer360_bronze.transactions_optimize_day14;

### 3. Hands-on: Safely inspect VACUUM candidates

In [0]:
%sql
DESCRIBE HISTORY
workspace.customer360_bronze.transactions_optimize_day14;

### Step 12 — Run VACUUM in DRY RUN mode

**DRY RUN** does not delete files. It lists files that are currently eligible for removal.

- Since your optimization was recent, you may see no eligible files under the default retention period.

In [0]:
%sql
VACUUM workspace.customer360_bronze.transactions_optimize_day14
DRY RUN;

### Step 13 — Check the retention configuration

In [0]:
%sql
SHOW TBLPROPERTIES
workspace.customer360_bronze.transactions_optimize_day14;

# 4. Understanding Delta Time Travel

### Step 14 — Query the current version

In [0]:
%sql
SELECT COUNT(*)
FROM workspace.customer360_bronze.transactions_optimize_day14;

### Step 15 — Query version 0

Expected: 100,000 records, assuming the original files are still available.

**Why does this work?**

Because OPTIMIZE changes the physical file layout but does not change the logical rows.

Therefore, both the original and optimized snapshots contain the same 100,000 transactions.

However, if a later VACUUM removes the original files, querying version 0 may fail even if its history entry remains visible.

In [0]:
%sql
SELECT COUNT(*)
FROM workspace.customer360_bronze.transactions_optimize_day14
VERSION AS OF 0;

- **OPTIMIZE** compacted your 40 files into one.
- The original 40 files became **obsolete but remain available** for historical reads.
- **VACUUM DRY RUN** found no eligible files to delete.
- Time Travel successfully accessed the original version.

**Recovery type-**
- A **full-table RESTORE** --> is appropriate when you intentionally want to revert the entire table. 
- **Selective recovery** --> is safer when valid changes have occurred after the incident.

# MINI CHALLENGE

## Production scenario — Accidental DELETE
Suppose a production table has these versions:
Version	Operation
100	Valid transaction data
101	Accidental DELETE
102	New valid transactions inserted
103	More valid updates

The business asks you to recover the deleted records.

**Option A — RESTORE the entire table**
````
RESTORE TABLE transactions
TO VERSION AS OF 100;
````

This restores the entire table to version 100.

Problem: It would also undo the valid changes made in versions 102 and 103.

**Option B — Selectively recover deleted records**
Instead, use Time Travel to identify the records that existed before the DELETE but are missing from the current table.

For example, assuming transactionId is a unique key:

````
INSERT INTO transactions
SELECT old.*
FROM transactions VERSION AS OF 100 AS old
LEFT ANTI JOIN transactions AS current
ON old.transactionId = current.transactionId
WHERE old.transactionDate = DATE '2026-11-01';
````


This illustrates selective recovery while preserving later inserts. In production, we would first validate the deleted keys, schema, and any legitimate subsequent changes before executing the INSERT.